## 1. Install Dependencies

In [17]:
!pip install sentence-transformers faiss-cpu ragas langchain langchain-community openai pymupdf python-dotenv tqdm --quiet

In [18]:
import yaml
from pathlib import Path

config = {
    "chunk_size": 300,
    "chunk_overlap": 50,
    "embed_model": "all-MiniLM-L6-v2",
    "top_k": 5
}

config_path = Path("..") / "configs" / "config.yaml"
config_path.parent.mkdir(parents=True, exist_ok=True)

with open(config_path, "w") as f:
    yaml.dump(config, f)

print(f" Config saved to {config_path}")

 Config saved to ..\configs\config.yaml


## 2. Imports & Config

In [19]:
import json, os, re
from pathlib import Path
from tqdm import tqdm
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
from dotenv import load_dotenv

load_dotenv(dotenv_path=Path("..") / ".env")

BASE_DIR       = Path("..")
PROCESSED_DIR  = BASE_DIR / "data" / "processed"
EXPERIMENTS    = BASE_DIR / "experiments"

# Load config
import yaml
with open(BASE_DIR / "configs" / "config.yaml") as f:
    config = yaml.safe_load(f)

CHUNK_SIZE    = config.get("chunk_size", 300)     # words per chunk
CHUNK_OVERLAP = config.get("chunk_overlap", 50)   # word overlap
EMBED_MODEL   = config.get("embed_model", "all-MiniLM-L6-v2")
TOP_K         = config.get("top_k", 5)

print(f" Config loaded")
print(f"   Chunk size    : {CHUNK_SIZE} words")
print(f"   Chunk overlap : {CHUNK_OVERLAP} words")
print(f"   Embed model   : {EMBED_MODEL}")
print(f"   Top-K         : {TOP_K}")

 Config loaded
   Chunk size    : 300 words
   Chunk overlap : 50 words
   Embed model   : all-MiniLM-L6-v2
   Top-K         : 5


## 3. Load Dataset from Notebook 01

In [20]:
with open(EXPERIMENTS / "01_exploration_summary.json") as f:
    dataset_log = json.load(f)

print(" Dataset summary from Notebook 01:")
print(json.dumps(dataset_log, indent=2))

# Re-run the PDF analysis to get text_data back in memory
import fitz
RAW_DIR = BASE_DIR / "data" / "raw"
pdfs = list(RAW_DIR.glob("*.pdf"))

def extract_digital_pages(pdf_path):
    doc = fitz.open(pdf_path)
    pages = []
    for page_num, page in enumerate(doc):
        text = page.get_text("text").strip()
        if text and len(text) > 50:
            pages.append({"source": Path(pdf_path).name, "page": page_num, "text": text})
    doc.close()
    return pages

all_pages = []
for pdf_path in pdfs:
    pages = extract_digital_pages(pdf_path)
    all_pages.extend(pages)
    print(f" {pdf_path.name}: {len(pages)} digital pages")

print(f"\n Total digital pages loaded: {len(all_pages)}")

 Dataset summary from Notebook 01:
{
  "total_pdfs": 5,
  "total_chunks": 114,
  "text_chunks": 93,
  "image_chunks": 21,
  "pdfs": {
    "Sample_1.pdf": {
      "type": "fully_scanned",
      "total_pages": 21,
      "digital": 0,
      "scanned": 21
    },
    "Sample_2.pdf": {
      "type": "fully_digital",
      "total_pages": 22,
      "digital": 22,
      "scanned": 0
    },
    "Sample_3.pdf": {
      "type": "fully_digital",
      "total_pages": 17,
      "digital": 17,
      "scanned": 0
    },
    "Sample_4.pdf": {
      "type": "fully_digital",
      "total_pages": 28,
      "digital": 28,
      "scanned": 0
    },
    "Sample_5.pdf": {
      "type": "fully_digital",
      "total_pages": 26,
      "digital": 26,
      "scanned": 0
    }
  }
}
 Sample_1.pdf: 0 digital pages
 Sample_2.pdf: 22 digital pages
 Sample_3.pdf: 17 digital pages
 Sample_4.pdf: 28 digital pages
 Sample_5.pdf: 26 digital pages

 Total digital pages loaded: 93


## 4. Text Chunking
Split each page into overlapping word-level chunks. Overlap preserves context at boundaries.

In [21]:
def chunk_text(text, source, page_num, chunk_size=300, overlap=50):
    words = text.split()
    chunks = []
    start = 0
    chunk_id = 0
    
    while start < len(words):
        end = min(start + chunk_size, len(words))
        chunk_words = words[start:end]
        chunks.append({
            "chunk_id": f"{source}_p{page_num}_c{chunk_id}",
            "source": source,
            "page": page_num,
            "text": " ".join(chunk_words),
            "word_count": len(chunk_words)
        })
        if end == len(words):
            break
        start += chunk_size - overlap
        chunk_id += 1
    
    return chunks

# Chunk all pages
all_chunks = []
for page in all_pages:
    chunks = chunk_text(page["text"], page["source"], page["page"],
                        chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP)
    all_chunks.extend(chunks)

word_counts = [c["word_count"] for c in all_chunks]
print(f" Total chunks created : {len(all_chunks)}")
print(f"   Avg words/chunk     : {int(np.mean(word_counts))}")
print(f"   Min / Max           : {min(word_counts)} / {max(word_counts)}")
print(f"\nSample chunk:")
print("-" * 50)
print(all_chunks[0]["text"][:300], "...")

 Total chunks created : 235
   Avg words/chunk     : 249
   Min / Max           : 49 / 300

Sample chunk:
--------------------------------------------------
Vol.:(0123456789) The Journal of Supercomputing (2024) 80:18325–18346 https://doi.org/10.1007/s11227-024-06153-2 1 3 A federated learning approach to network intrusion detection using residual networks in industrial IoT networks Nisha Chaurasia1 · Munna Ram1 · Priyanka Verma3 · Nakul Mehta1 · Nitesh ...


## 5. Generate Embeddings
`all-MiniLM-L6-v2` — fast, 384-dim, great for semantic search on research text.

In [22]:
print(f" Loading model: {EMBED_MODEL}")
embedder = SentenceTransformer(EMBED_MODEL)

texts = [c["text"] for c in all_chunks]
print(f" Embedding {len(texts)} chunks...")

embeddings = embedder.encode(
    texts,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True
)

print(f"\n Embeddings shape : {embeddings.shape}")
print(f"   Dtype            : {embeddings.dtype}")

 Loading model: all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


 Embedding 235 chunks...


Batches:   0%|          | 0/4 [00:00<?, ?it/s]


 Embeddings shape : (235, 384)
   Dtype            : float32


## 6. Build FAISS Index
Using `IndexFlatIP` (inner product) with normalized vectors = cosine similarity.

In [23]:
# Normalize for cosine similarity
norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
embeddings_norm = embeddings / norms

dim = embeddings_norm.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(embeddings_norm)

print(f" FAISS index built")
print(f"   Vectors indexed : {index.ntotal}")
print(f"   Dimensions      : {dim}")

# Save index + chunks metadata
VECTOR_DIR = PROCESSED_DIR / "vectors"
VECTOR_DIR.mkdir(parents=True, exist_ok=True)

faiss.write_index(index, str(VECTOR_DIR / "text_index.faiss"))

with open(VECTOR_DIR / "text_chunks.json", "w") as f:
    json.dump(all_chunks, f, indent=2)

print(f"\n Saved to: {VECTOR_DIR}")
print("   text_index.faiss")
print("   text_chunks.json")

 FAISS index built
   Vectors indexed : 235
   Dimensions      : 384

 Saved to: ..\data\processed\vectors
   text_index.faiss
   text_chunks.json


## 7. Retrieval — Manual Smoke Test
Before evaluation, manually verify retrieval quality with a few queries.

In [24]:
def retrieve(query, index, chunks, embedder, top_k=5):
    q_emb = embedder.encode([query], convert_to_numpy=True)
    q_emb = q_emb / np.linalg.norm(q_emb, axis=1, keepdims=True)
    
    scores, indices = index.search(q_emb, top_k)
    
    results = []
    for score, idx in zip(scores[0], indices[0]):
        results.append({
            "chunk_id": chunks[idx]["chunk_id"],
            "source": chunks[idx]["source"],
            "page": chunks[idx]["page"],
            "score": round(float(score), 4),
            "text": chunks[idx]["text"]
        })
    return results

# Test queries — adjust to match your actual PDFs
test_queries = [
    "What is federated learning?",
    "How does gradient aggregation work in federated settings?",
    "Privacy challenges in distributed machine learning"
]

for query in test_queries:
    print(f"\n Query: {query}")
    print("-" * 60)
    results = retrieve(query, index, all_chunks, embedder, top_k=TOP_K)
    for i, r in enumerate(results[:3]):
        print(f"  [{i+1}] Score: {r['score']} | {r['source']} p.{r['page']+1}")
        print(f"       {r['text'][:150]}...")


 Query: What is federated learning?
------------------------------------------------------------
  [1] Score: 0.684 | Sample_2.pdf p.19
       18343 1 3 A federated learning approach to network intrusion detection… its benefits, some concerns need to be addressed. One of the main issues is th...
  [2] Score: 0.6732 | Sample_5.pdf p.16
       Fig. 8, and deep neural networks to increase the success of RL in complex, high-dimensional environments, such as arcade games, video games, or roboti...
  [3] Score: 0.5834 | Sample_2.pdf p.12
       18336 N. Chaurasia et al. 1 3 average pooling, and sum pooling. In max pooling, the feature map’s largest element is chosen, whereas in average poolin...

 Query: How does gradient aggregation work in federated settings?
------------------------------------------------------------
  [1] Score: 0.4894 | Sample_2.pdf p.12
       18336 N. Chaurasia et al. 1 3 average pooling, and sum pooling. In max pooling, the feature map’s largest element is chosen, 

## 8. Evaluation with RAGAS
Create a small ground-truth QA set and measure:
- **Context Precision** — are retrieved chunks relevant?
- **Context Recall** — are all relevant chunks retrieved?
- **Answer Relevancy** — does the answer address the question?

>  RAGAS requires an LLM for answer generation. Set `OPENAI_API_KEY` in `.env` or swap in a local LLM.

In [25]:
# ---- Build a mini eval set (20-30 pairs for real eval) ----
# Replace these with questions answerable from YOUR PDFs
eval_questions = [
    "What aggregation algorithm is used in federated learning?",
    "What are the privacy guarantees discussed in the paper?",
    "How does communication overhead affect federated training?",
]

# Retrieve contexts for each question
eval_dataset = []
for q in eval_questions:
    results = retrieve(q, index, all_chunks, embedder, top_k=TOP_K)
    contexts = [r["text"] for r in results]
    eval_dataset.append({"question": q, "contexts": contexts})
    print(f" Retrieved {len(contexts)} contexts for: {q[:50]}")

 Retrieved 5 contexts for: What aggregation algorithm is used in federated le
 Retrieved 5 contexts for: What are the privacy guarantees discussed in the p
 Retrieved 5 contexts for: How does communication overhead affect federated t


In [26]:
# ---- Run RAGAS (context precision/recall — no LLM needed) ----
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import context_precision, context_recall

# NOTE: context_recall needs ground_truth. Add manually for real eval.
# For a quick smoke test, we measure context_precision only.

ragas_data = Dataset.from_list([
    {"question": d["question"], "contexts": d["contexts"], "answer": ""}
    for d in eval_dataset
])

# results = evaluate(ragas_data, metrics=[context_precision])
# print(results)

#  Uncomment above once OPENAI_API_KEY is set in .env
# For now, print contexts to manually inspect quality
print(" Manual context inspection:")
for item in eval_dataset:
    print(f"\nQ: {item['question']}")
    print(f"Top context: {item['contexts'][0][:200]}...")

 Manual context inspection:

Q: What aggregation algorithm is used in federated learning?
Top context: 18336 N. Chaurasia et al. 1 3 average pooling, and sum pooling. In max pooling, the feature map’s largest element is chosen, whereas in average pooling, the output is the average of all values. Sum po...

Q: What are the privacy guarantees discussed in the paper?
Top context: network management, it’s essential to evaluate the potential impacts of their use. Privacy and Data Protection 1. Data Collection and Storage: AI algorithms need extensive data for learning and improv...

Q: How does communication overhead affect federated training?
Top context: 18343 1 3 A federated learning approach to network intrusion detection… its benefits, some concerns need to be addressed. One of the main issues is that it may not be secure against federated learning...


C:\Users\Naveen Reddie\AppData\Local\Temp\ipykernel_43452\4102274137.py:4: DeprecationWarning: Importing context_precision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_precision
  from ragas.metrics import context_precision, context_recall
C:\Users\Naveen Reddie\AppData\Local\Temp\ipykernel_43452\4102274137.py:4: DeprecationWarning: Importing context_recall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_recall
  from ragas.metrics import context_precision, context_recall


## 9. Save Baseline Results

In [27]:
baseline_results = {
    "notebook": "02_text_rag_baseline",
    "embed_model": EMBED_MODEL,
    "chunk_size": CHUNK_SIZE,
    "chunk_overlap": CHUNK_OVERLAP,
    "total_chunks": len(all_chunks),
    "index_vectors": index.ntotal,
    "top_k": TOP_K,
    "notes": "Text-only RAG baseline. Multimodal extension in Notebook 03."
}

with open(EXPERIMENTS / "02_text_rag_baseline.json", "w") as f:
    json.dump(baseline_results, f, indent=2)

print(" Baseline results saved to experiments/02_text_rag_baseline.json")
print(json.dumps(baseline_results, indent=2))


 Baseline results saved to experiments/02_text_rag_baseline.json
{
  "notebook": "02_text_rag_baseline",
  "embed_model": "all-MiniLM-L6-v2",
  "chunk_size": 300,
  "chunk_overlap": 50,
  "total_chunks": 235,
  "index_vectors": 235,
  "top_k": 5,
  "notes": "Text-only RAG baseline. Multimodal extension in Notebook 03."
}
